# 3-1 OpenCV Test

This notebook aligns the realtime OpenCV flow with `3_compare/1-7_compare.ipynb`.


In [1]:
from pathlib import Path
import time

import cv2
import mediapipe as mp
import numpy as np
import torch
from torch import nn
from PIL import Image
from torchvision import models, transforms

model_dir = Path("models")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def make_eval_transform(image_size: int):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def change_fc(model, class_count: int):
    model.fc = nn.Linear(model.fc.in_features, class_count)
    return model

def change_classifier(model, class_count: int):
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, class_count)
    return model

model_list = {
    "resnet18": [models.resnet18, change_fc],
    "resnet34": [models.resnet34, change_fc],
    "resnet50": [models.resnet50, change_fc],
    "mobilenet_v3_large": [models.mobilenet_v3_large, change_classifier],
    "efficientnet_b0": [models.efficientnet_b0, change_classifier],
    "efficientnet_b2": [models.efficientnet_b2, change_classifier],
    "efficientnet_b3": [models.efficientnet_b3, change_classifier],
    "efficientnet_v2_s": [models.efficientnet_v2_s, change_classifier],
}

def infer_model_name(path: Path, checkpoint: dict):
    if "model_name" in checkpoint:
        return checkpoint["model_name"]
    return path.stem.removesuffix("_best")

def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def get_class_names(checkpoint: dict):
    if "class_names" in checkpoint:
        return list(checkpoint["class_names"])
    if "class_to_idx" in checkpoint:
        return [name for name, _ in sorted(checkpoint["class_to_idx"].items(), key=lambda item: item[1])]
    raise KeyError("checkpoint is missing class_names or class_to_idx")

def get_model_state(checkpoint: dict):
    if "model_state" in checkpoint:
        return checkpoint["model_state"]
    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    raise KeyError("checkpoint is missing model_state or model_state_dict")

def create_model(model_name: str, class_count: int):
    if model_name not in model_list:
        raise ValueError(f"unsupported model name: {model_name}")

    model_func, change_last_layer = model_list[model_name]
    model = model_func(weights=None)
    model = change_last_layer(model, class_count)
    return model


device: cuda


## Realtime OpenCV Test

Check `realtime_model_name` and run the next cell.

- Examples: `resnet18`, `resnet34`, `resnet50`, `mobilenet_v3_large`, `efficientnet_b0`, `efficientnet_b2`, `efficientnet_b3`, `efficientnet_v2_s`
- Press `q` in the OpenCV window to quit.


In [2]:
from collections import deque

realtime_model_name = "resnet18"

camera_index = 0
frame_width = 960
frame_height = 720
window_name = f"Realtime Focus Detection - {realtime_model_name}"

face_model_path = Path("mp_model/face_landmarker.task")
face_padding_ratio = 0.25
probability_window_size = 10
confidence_threshold = 0.50
margin_threshold = 0.10

def load_model_for_realtime(model_name: str):
    model_path = model_dir / f"{model_name}_best.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"model file not found: {model_path}")

    checkpoint = load_checkpoint(model_path)
    checkpoint_model_name = infer_model_name(model_path, checkpoint)
    class_names = get_class_names(checkpoint)
    image_size = int(checkpoint.get("image_size", 224))

    model = create_model(checkpoint_model_name, len(class_names)).to(device)
    model.load_state_dict(get_model_state(checkpoint))
    model.eval()

    idx_to_class = {idx: name for idx, name in enumerate(class_names)}
    transform = make_eval_transform(image_size)
    return model, class_names, idx_to_class, transform, image_size, model_path

def load_realtime_face_landmarker():
    if not face_model_path.exists():
        raise FileNotFoundError(f"MediaPipe face model not found: {face_model_path}")

    options = mp.tasks.vision.FaceLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(face_model_path)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp.tasks.vision.FaceLandmarker.create_from_options(options)

def realtime_face_landmarks_to_bbox(face_landmarks, frame_width: int, frame_height: int):
    xs = [int(lm.x * frame_width) for lm in face_landmarks]
    ys = [int(lm.y * frame_height) for lm in face_landmarks]
    x1 = max(0, min(xs))
    y1 = max(0, min(ys))
    x2 = min(frame_width - 1, max(xs))
    y2 = min(frame_height - 1, max(ys))
    return x1, y1, x2, y2

def crop_face_for_realtime(frame_bgr, face_box, output_size: int, padding_ratio: float = face_padding_ratio):
    if face_box is None:
        return None

    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = face_box
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)
    pad_x = int(box_w * padding_ratio)
    pad_y = int(box_h * padding_ratio)

    crop_x1 = max(0, x1 - pad_x)
    crop_y1 = max(0, y1 - pad_y)
    crop_x2 = min(width - 1, x2 + pad_x)
    crop_y2 = min(height - 1, y2 + pad_y)

    crop = frame_bgr[crop_y1:crop_y2 + 1, crop_x1:crop_x2 + 1]
    if crop.size == 0:
        return None

    return cv2.resize(crop, (output_size, output_size), interpolation=cv2.INTER_AREA)

def predict_realtime_crop(face_crop_bgr, model, transform):
    face_crop_rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(face_crop_rgb)
    inputs = transform(pil_image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(inputs)
        return torch.softmax(outputs, dim=1).squeeze(0).cpu()

def choose_realtime_status(avg_probs: torch.Tensor, idx_to_class: dict[int, str]):
    top_probs, top_indices = torch.topk(avg_probs, k=min(2, len(avg_probs)))
    pred_idx = int(top_indices[0].item())
    pred_class = idx_to_class[pred_idx]
    pred_prob = float(top_probs[0].item())

    margin = pred_prob
    if len(top_probs) >= 2:
        margin = float(top_probs[0].item() - top_probs[1].item())

    if pred_prob < confidence_threshold or margin < margin_threshold:
        return "Uncertain", pred_prob
    return pred_class, pred_prob

def draw_realtime_overlay(frame_bgr, face_box, display_class, display_prob, display_probs, fps):
    if face_box is not None:
        x1, y1, x2, y2 = face_box
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 255), 2)

    lines = [
        f"Model: {realtime_model_name}",
        f"FPS: {fps:.1f}",
        f"Pred: {display_class} ({display_prob:.2f})",
    ]
    for class_name, prob in display_probs.items():
        lines.append(f"{class_name}: {prob:.2f}")
    lines.append("q: Quit")

    y = 32
    for line in lines:
        color = (0, 255, 0) if line.startswith("Pred:") else (255, 255, 255)
        cv2.putText(frame_bgr, line, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
        y += 30
    return frame_bgr

model, class_names, idx_to_class, realtime_transform, image_size, loaded_model_path = load_model_for_realtime(realtime_model_name)
display_probs = {class_name: 0.0 for class_name in class_names}
probability_window = deque(maxlen=probability_window_size)
display_class = "Waiting"
display_prob = 0.0

print("loaded model:", loaded_model_path)
print("classes:", class_names)
print("image_size:", image_size)
print("Press q in the OpenCV window to quit.")

cv2.destroyAllWindows()

with load_realtime_face_landmarker() as face_landmarker:
    camera_capture = cv2.VideoCapture(camera_index)
    camera_capture.set(cv2.CAP_PROP_FRAME_WIDTH, frame_width)
    camera_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, frame_height)

    if not camera_capture.isOpened():
        raise RuntimeError("Camera open failed. Check camera_index and camera permission.")

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    start_time = time.perf_counter()
    last_frame_time = time.perf_counter()

    try:
        while True:
            is_readable, frame_bgr = camera_capture.read()
            if not is_readable:
                print("Frame read failed.")
                break

            frame_bgr = cv2.flip(frame_bgr, 1)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame_rgb))
            timestamp_ms = int((time.perf_counter() - start_time) * 1000)

            face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)ㅂ
            face_landmarks_list = getattr(face_result, "face_landmarks", [])
            face_box = None

            if face_landmarks_list:
                face_box = realtime_face_landmarks_to_bbox(face_landmarks_list[0], frame_bgr.shape[1], frame_bgr.shape[0])
                face_crop_bgr = crop_face_for_realtime(frame_bgr, face_box, image_size)
                if face_crop_bgr is not None:
                    probs = predict_realtime_crop(face_crop_bgr, model, realtime_transform)
                    probability_window.append(probs)

                if probability_window:
                    avg_probs = torch.stack(list(probability_window)).mean(dim=0)
                    display_probs = {
                        idx_to_class[idx]: float(avg_probs[idx].item())
                        for idx in range(len(idx_to_class))
                    }
                    display_class, display_prob = choose_realtime_status(avg_probs, idx_to_class)
            else:
                probability_window.clear()
                display_class = "No Face"
                display_prob = 0.0
                display_probs = {class_name: 0.0 for class_name in class_names}

            current_time = time.perf_counter()
            fps = 1.0 / max(1e-6, current_time - last_frame_time)
            last_frame_time = current_time

            display_frame_bgr = draw_realtime_overlay(
                frame_bgr,
                face_box,
                display_class,
                display_prob,
                display_probs,
                fps,
            )
            cv2.imshow(window_name, display_frame_bgr)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                print("Quit")
                break
    finally:
        camera_capture.release()
        cv2.destroyAllWindows()


loaded model: models\resnet18_best.pt
classes: ['Attentive', 'Drowsy', 'LookingAway']
image_size: 224
Press q in the OpenCV window to quit.
Quit
